In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expi
from scipy.optimize import curve_fit

"""
Herramientas Hidrogeológicas
Pruebas de bombeo en acuiferos confinado (Método de Theis)

Descripción: Ajuste inverso automatizado para estimar la Transmisividad (T) y el Coeficiente de Almacenamiento (S) 
mediante la superposición de la curva patrón de Theis.

Autor: Carlos Javier Pérez Pérez
"""

In [ ]:
# 1. Datos de campo y parámetros iniciales
# Leer el archivo de Excel
df = pd.read_excel('datos_campo_theis.xlsx')

caudal_lps = 20.0       
radio_obs_m = 150.0

# Conversión a unidades estándar (m³/día y días)
caudal_m3d = caudal_lps * 86.4 
df['tiempo_dias'] = df['tiempo_min'] / 1440.0

In [ ]:
# 2. Definición del modelo analítico
def modelo_theis(t_dias: float, T: float, S: float) -> float:
    """Calcula el descenso teórico usando la Función de Pozo de Theis."""
    u = (radio_obs_m**2 * S) / (4 * T * t_dias)
    W_u = -expi(-u)
    descenso = (caudal_m3d / (4 * np.pi * T)) * W_u
    return descenso

In [ ]:
# 3. Calibración (Ajuste inverso)
estimacion_inicial = [100.0, 0.0001]
(T_calc, S_calc), matriz_covarianza = curve_fit(
    modelo_theis, 
    df['tiempo_dias'], 
    df['descenso_m'], 
    p0=estimacion_inicial
)

In [ ]:
# 4. Reconstrucción del Punto de Ajuste tradicional (W(u)=1, 1/u=10)
u_ajuste = 0.1
w_u_ajuste = 1.0

s_ajuste = (caudal_m3d / (4 * np.pi * T_calc)) * w_u_ajuste
t_ajuste_dias = (radio_obs_m**2 * S_calc) / (4 * T_calc * u_ajuste)
t_ajuste_min = t_ajuste_dias * 1440.0

In [ ]:
# 5. Reporte en consola
print("Reporte Hidrogeológico: Método De Theis")
print("-" * 45)
print(f"Transmisividad (T)     : {T_calc:.0f} m²/día")
print(f"Almacenamiento (S)     : {S_calc:.1e}")
print(f"Punto De Ajuste Teórico: t = {t_ajuste_min:.1f} min, s = {s_ajuste:.2f} m\n")

In [ ]:
# 6. Configuración y exportación del gráfico de diagnóstico
df['1/u'] = (4 * T_calc * df['tiempo_dias']) / (radio_obs_m**2 * S_calc)
df['W(u)'] = df['descenso_m'] * (4 * np.pi * T_calc) / caudal_m3d

u_teorico = np.logspace(-2, 5, 100)
w_teorico = -expi(-(1/u_teorico))

plt.rcParams.update({'font.family': 'serif', 'font.size': 11})
fig, ax = plt.subplots(figsize=(10, 6))

# Trazado geométrico y limpio de las series
ax.loglog(u_teorico, w_teorico, color='#2c3e50', linewidth=2, label='Curva Patrón: $W(u)$ vs $1/u$')
ax.loglog(df['1/u'], df['W(u)'], 's', color='#b30000', markersize=8, label='Medidas De Campo')
ax.loglog(10, 1, 'P', color='#27ae60', markersize=14, label='Punto De Ajuste (10, 1)')

ax.grid(True, which="both", ls="--", alpha=0.5, color='gray')
ax.set_xlabel('$1/u$ (Proporcional Al Tiempo)', fontweight='bold')
ax.set_ylabel('$W(u)$ (Proporcional Al Descenso)', fontweight='bold')
ax.set_title('Análisis De Prueba De Bombeo: Método De Theis', pad=15, fontweight='bold')

# Líneas guía
ax.axhline(1, color='#27ae60', linestyle=':', alpha=0.5)
ax.axvline(10, color='#27ae60', linestyle=':', alpha=0.5)

# Caja de resultados
texto_resultados = (
    f"Parámetros Estimados:\n"
    f"T = {T_calc:.0f} m²/d\n"
    f"S = {S_calc:.1e}"
)
props_caja = dict(boxstyle='square,pad=0.6', facecolor='#f9f9f9', edgecolor='black', alpha=0.9)
ax.text(0.04, 0.96, texto_resultados, transform=ax.transAxes, fontsize=10, verticalalignment='top', bbox=props_caja)

ax.legend(loc='lower center', framealpha=1.0, edgecolor='black')
plt.tight_layout()
plt.show()